In [ ]:
import re
import pandas as pd

# ─────────────────────────────────────────────
# CONFIGURACIÓN
# ─────────────────────────────────────────────
INPUT_FILE  = "inventario_concesionario.csv"
OUTPUT_FILE = "inventario_limpio.csv"

# ─────────────────────────────────────────────
# PASO 1 — EXTRACT
# ─────────────────────────────────────────────
def extract(path: str) -> pd.DataFrame:
    """Lee el CSV crudo y devuelve un DataFrame sin modificaciones."""
    df = pd.read_csv(path, dtype=str)           # dtype=str para no perder errores de formato
    df.columns = df.columns.str.strip()         # elimina espacios en los nombres de columna
    print(f"[EXTRACT] {len(df)} filas cargadas desde '{path}'")
    print(f"          Columnas: {list(df.columns)}\n")
    return df


# ─────────────────────────────────────────────
# PASO 2 — TRANSFORM  (una función por columna)
# ─────────────────────────────────────────────

# ── 2.1  MARCA ───────────────────────────────
MARCAS_CORRECTAS = {
    "toyta"      : "Toyota",
    "toyota"     : "Toyota",
    "hnda"       : "Honda",
    "honda"      : "Honda",
    "ford"       : "Ford",
    "chevrolet"  : "Chevrolet",
    "nissan"     : "Nissan",
    "hyundai"    : "Hyundai",
    "kia"        : "Kia",
    "volkswagen" : "Volkswagen",
}

def limpiar_marca(serie: pd.Series) -> pd.Series:
    return (serie
            .str.strip()
            .str.lower()
            .map(MARCAS_CORRECTAS)            # mapea errores tipográficos
            )


# ── 2.2  ESTADO ──────────────────────────────
ESTADOS_CORRECTOS = {
    "disponible"  : "disponible",
    "disponbile"  : "disponible",
    "vendido"     : "vendido",
    "vendidoo"    : "vendido",
    "taller"      : "taller",
}

def limpiar_estado(serie: pd.Series) -> pd.Series:
    return (serie
            .str.strip()
            .str.lower()
            .map(ESTADOS_CORRECTOS)
            )


# ── 2.3  AÑO ─────────────────────────────────
AÑOS_EN_TEXTO = {
    "dos mil quince"      : "2015",
    "dos mil dieciséis"   : "2016",
    "dos mil dieciseis"   : "2016",
    "dos mil diecisiete"  : "2017",
    "dos mil dieciocho"   : "2018",
    "dos mil diecinueve"  : "2019",
    "dos mil veinte"      : "2020",
    "dos mil veintiuno"   : "2021",
    "dos mil veintidós"   : "2022",
    "dos mil veintidos"   : "2022",
}

def limpiar_año(serie: pd.Series) -> pd.Series:
    def _fix(val):
        if pd.isna(val):
            return pd.NA
        val = str(val).strip().lower()
        # texto en español → número
        if val in AÑOS_EN_TEXTO:
            val = AÑOS_EN_TEXTO[val]
        # reemplaza letra O mayúscula/minúscula por cero dentro de dígitos  (ej: "201O")
        val = re.sub(r'[Oo]', '0', val)
        # conserva solo dígitos
        val = re.sub(r'\D', '', val)
        if len(val) == 4:
            return int(val)
        return pd.NA                           # año irreparable → NaN

    return serie.map(_fix).astype("Int64")    # Int64 soporta NaN enteros


# ── 2.4  KILOMETRAJE ─────────────────────────
def limpiar_kilometraje(serie: pd.Series) -> pd.Series:
    def _fix(val):
        if pd.isna(val):
            return pd.NA
        val = str(val).strip().lower()
        # "70k" → 70000
        if val.endswith('k'):
            try:
                return int(float(val[:-1]) * 1_000)
            except ValueError:
                return pd.NA
        # elimina espacios internos  ("100 000" → "100000")
        val = val.replace(' ', '')
        try:
            return int(float(val))
        except ValueError:
            return pd.NA

    return serie.map(_fix).astype("Int64")


# ── 2.5  MOTOR ───────────────────────────────
def limpiar_motor(serie: pd.Series) -> pd.Series:
    def _fix(val):
        if pd.isna(val):
            return pd.NA
        val = str(val).strip()
        # reemplaza "O" (letra) por "0" después de punto decimal  (ej: "2.O")
        val = re.sub(r'(?<=\d\.)([Oo])', '0', val)
        try:
            return float(val)
        except ValueError:
            return pd.NA

    return serie.map(_fix)


# ── 2.6  PLACA ───────────────────────────────
PATRON_PLACA_VALIDO = re.compile(r'^[A-Z]{3}\d{3}$')

def limpiar_placa(serie: pd.Series) -> pd.Series:
    def _fix(val):
        if pd.isna(val):
            return pd.NA
        val = str(val).strip().upper()
        # elimina guiones y espacios para intentar recuperar
        val_limpio = re.sub(r'[-\s]', '', val)
        if PATRON_PLACA_VALIDO.match(val_limpio):
            return val_limpio
        # placa invertida tipo "123ABC" → intenta reorganizar
        m = re.match(r'^(\d{3})([A-Z]{3})$', val_limpio)
        if m:
            return m.group(2) + m.group(1)    # "123ABC" → "ABC123"
        return None                            # formato irreparable

    return serie.map(_fix)


# ── 2.7  CV (caballos de potencia) ───────────
CV_EN_TEXTO = {
    "cien"            : 100,
    "ciento diez"     : 110,
    "ciento veinte"   : 120,
    "ciento treinta"  : 130,
    "ciento cuarenta" : 140,
    "ciento cincuenta": 150,
}

def limpiar_cv(serie: pd.Series) -> pd.Series:
    def _fix(val):
        if pd.isna(val):
            return pd.NA
        val_str = str(val).strip().lower()
        if val_str in CV_EN_TEXTO:
            return CV_EN_TEXTO[val_str]
        try:
            return int(float(val_str))
        except ValueError:
            return pd.NA

    return serie.map(_fix).astype("Int64")


# ── 2.8  PESO_KG ─────────────────────────────
def limpiar_peso(serie: pd.Series) -> pd.Series:
    def _fix(val):
        if pd.isna(val):
            return pd.NA
        val = str(val).strip()
        # "1.200" con punto como separador de miles → elimina el punto
        # Heurística: si hay un punto pero no es decimal estándar (más de 3 dígitos tras el punto)
        if re.match(r'^\d{1,2}\.\d{3}$', val):
            val = val.replace('.', '')
        try:
            return float(val)
        except ValueError:
            return pd.NA

    return serie.map(_fix)


# ── 2.9  TRANSMISIÓN ─────────────────────────
TRANSMISIONES_CORRECTAS = {
    "manual"    : "manual",
    "manul"     : "manual",
    "automatica": "automatica",
    "automática": "automatica",
    "auto"      : "automatica",
}

def limpiar_transmision(serie: pd.Series) -> pd.Series:
    return (serie
            .str.strip()
            .str.lower()
            .map(TRANSMISIONES_CORRECTAS)
            )


# ── 2.10  PRECIO_COP ─────────────────────────
MILLONES_TEXTO = re.compile(
    r'(\d+[\.,]?\d*)\s*millones?', re.IGNORECASE
)

def limpiar_precio(serie: pd.Series) -> pd.Series:
    def _fix(val):
        if pd.isna(val):
            return pd.NA
        val = str(val).strip()
        if val == '':
            return pd.NA

        # "45 millones" → 45000000
        m = MILLONES_TEXTO.match(val)
        if m:
            num = m.group(1).replace(',', '.').replace(' ', '')
            try:
                return int(float(num) * 1_000_000)
            except ValueError:
                return pd.NA

        # "40.000.000" con puntos como separadores de miles
        # Detecta cuando hay varios puntos (no es decimal)
        if val.count('.') > 1:
            val = val.replace('.', '')

        # elimina espacios internos y finales
        val = val.replace(' ', '')

        try:
            return int(float(val))
        except ValueError:
            return pd.NA

    return serie.map(_fix).astype("Int64")


# ── Pipeline completo ─────────────────────────
def transform(df: pd.DataFrame) -> pd.DataFrame:
    print("[TRANSFORM] Iniciando limpieza de columnas...")

    df_limpio = df.copy()

    df_limpio["marca"]       = limpiar_marca(df["marca"])
    df_limpio["estado"]      = limpiar_estado(df["estado"])
    df_limpio["año"]         = limpiar_año(df["año"])
    df_limpio["kilometraje"] = limpiar_kilometraje(df["kilometraje"])
    df_limpio["motor"]       = limpiar_motor(df["motor"])
    df_limpio["placa"]       = limpiar_placa(df["placa"])
    df_limpio["cv"]          = limpiar_cv(df["cv"])
    df_limpio["peso_kg"]     = limpiar_peso(df["peso_kg"])
    df_limpio["transmisión"] = limpiar_transmision(df["transmisión"])
    df_limpio["precio_cop"]  = limpiar_precio(df["precio_cop"])

    # ── Reporte de calidad ────────────────────
    print("\n[TRANSFORM] Resumen de valores nulos tras limpieza:")
    nulos = df_limpio.isna().sum()
    for col, n in nulos[nulos > 0].items():
        print(f"  · {col:<15} → {n} nulo(s)")

    print(f"\n[TRANSFORM] Registros con placa inválida (None): "
          f"{df_limpio['placa'].isna().sum()}")

    return df_limpio


# ─────────────────────────────────────────────
# PASO 3 — LOAD
# ─────────────────────────────────────────────
def load(df: pd.DataFrame, path: str) -> None:
    """Guarda el DataFrame limpio como CSV."""
    df.to_csv(path, index=False, encoding="utf-8-sig")
    print(f"\n[LOAD] Archivo limpio guardado en '{path}'")
    print(f"       {len(df)} filas · {len(df.columns)} columnas")


# ─────────────────────────────────────────────
# MAIN
# ─────────────────────────────────────────────
def main():
    print("=" * 52)
    print("  ETL — Inventario Concesionario de Autos - DriveAnalytics")
    print("=" * 52 + "\n")

    # E → T → L
    df_raw   = extract(INPUT_FILE)
    df_clean = transform(df_raw)
    load(df_clean, OUTPUT_FILE)

    # Vista previa final
    print("\n[PREVIEW] Primeras 5 filas del dataset limpio:")
    print(df_clean.head().to_string(index=False))
    print("\n[PREVIEW] Tipos de datos finales:")
    print(df_clean.dtypes.to_string())
    print("\n✅  ETL completado exitosamente.")


if __name__ == "__main__":
    main()

  ETL — Inventario Concesionario de Autos

[EXTRACT] 99 filas cargadas desde 'inventario_concesionario.csv'
          Columnas: ['marca', 'modelo', 'año', 'estado', 'kilometraje', 'motor', 'placa', 'cv', 'peso_kg', 'transmisión', 'precio_cop']

[TRANSFORM] Iniciando limpieza de columnas...

[TRANSFORM] Resumen de valores nulos tras limpieza:
  · placa           → 1 nulo(s)
  · precio_cop      → 1 nulo(s)

[TRANSFORM] Registros con placa inválida (None): 1

[LOAD] Archivo limpio guardado en 'inventario_limpio.csv'
       99 filas · 11 columnas

[PREVIEW] Primeras 5 filas del dataset limpio:
    marca  modelo  año     estado  kilometraje  motor  placa  cv  peso_kg transmisión  precio_cop
   Toyota Corolla 2018 disponible        45000    1.8 ABC123 140   1250.0      manual    52000000
    Honda   Civic 2020    vendido        20000    1.5 DEF456 158   1300.0  automatica    68000000
     Ford  Fiesta 2017     taller        60000    1.6 GHI789 120   1100.0      manual    38000000
Chevrolet  